# Notebook 03 (Participant): Add a New Problem Scaffold

You will implement a robotics co-design problem contract and evaluate whether it is benchmark-ready.


**Edit-safe start:** this notebook opens from GitHub in read-only source mode. Use **File -> Save a copy in Drive** before running edits so your changes stay in your own workspace.


## Notebook map

This notebook is written as a standalone lab chapter:
- context first,
- implementation second,
- interpretation third.

If you are following asynchronously, run cells in order and use the success checks to validate each stage before moving on.


## Standalone guide

This chapter is about benchmark design quality, not model training speed.


## What makes a new problem benchmark-ready

A publishable benchmark needs explicit representation, constraints, objectives, simulator semantics, and reproducibility metadata.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'matplotlib', 'gymnasium', 'pybullet']

if IN_COLAB or FORCE_INSTALL:
    print('Installing dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


### Step 1 - Import scaffold dependencies

Ensure all required interfaces are visible before class implementation.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Annotated

import numpy as np
from gymnasium import spaces

from engibench.constraint import bounded
from engibench.constraint import constraint
from engibench.core import ObjectiveDirection
from engibench.core import OptiStep
from engibench.core import Problem

import pybullet as p


### Step 2 - Implement PyBullet manipulator co-design problem contract (TODO)

Complete each required method with deterministic behavior and clear failure messages.


In [ ]:
class PlanarManipulatorCoDesignProblem(Problem[np.ndarray]):
    """Robotics co-design scaffold using a real PyBullet rollout loop."""

    version = 0
    objectives = (
        ("final_tracking_error_m", ObjectiveDirection.MINIMIZE),
        ("actuation_energy_j", ObjectiveDirection.MINIMIZE),
    )

    @dataclass
    class Conditions:
        target_x: Annotated[float, bounded(lower=0.20, upper=1.35)] = 0.85
        target_y: Annotated[float, bounded(lower=0.05, upper=1.20)] = 0.45
        payload_kg: Annotated[float, bounded(lower=0.0, upper=2.0)] = 0.8
        disturbance_scale: Annotated[float, bounded(lower=0.0, upper=0.30)] = 0.05

    @dataclass
    class Config(Conditions):
        sim_steps: Annotated[int, bounded(lower=60, upper=1200)] = 240
        dt: Annotated[float, bounded(lower=1e-4, upper=0.05)] = 1.0 / 120.0
        torque_limit: Annotated[float, bounded(lower=1.0, upper=50.0)] = 12.0
        max_iter: Annotated[int, bounded(lower=1, upper=300)] = 60

    dataset_id = "IDEALLab/planar_manipulator_codesign_v0"  # placeholder for future dataset integration
    container_id = None

    def __init__(self, seed: int = 0, **kwargs):
        super().__init__(seed=seed)
        self.config = self.Config(**kwargs)
        self.conditions = self.Conditions(
            target_x=self.config.target_x,
            target_y=self.config.target_y,
            payload_kg=self.config.payload_kg,
            disturbance_scale=self.config.disturbance_scale,
        )

        # Design vector = [link1_m, link2_m, motor_strength, kp, kd, damping]
        self.design_space = spaces.Box(
            low=np.array([0.25, 0.20, 2.0, 5.0, 0.2, 0.0], dtype=np.float32),
            high=np.array([1.00, 0.95, 30.0, 120.0, 18.0, 1.5], dtype=np.float32),
            dtype=np.float32,
        )

        # TODO 1: implement design constraints and assign self.design_constraints
        # Suggested constraints:
        # - reachable workspace: link1+link2 must reach target radius
        # - gain consistency: kd must be bounded relative to kp
        raise NotImplementedError('Implement __init__ constraints')

    def _build_robot(self, l1: float, l2: float, payload_kg: float, damping: float) -> tuple[int, int]:
        # TODO 2: build 2-link planar robot in PyBullet DIRECT mode
        raise NotImplementedError('Implement _build_robot')

    def _inverse_kinematics_2link(self, x: float, y: float, l1: float, l2: float) -> tuple[float, float]:
        # TODO 3: implement closed-form IK for 2-link planar arm
        raise NotImplementedError('Implement _inverse_kinematics_2link')

    def _forward_kinematics_2link(self, q1: float, q2: float, l1: float, l2: float) -> tuple[float, float]:
        # TODO 4: implement FK for end-effector position
        raise NotImplementedError('Implement _forward_kinematics_2link')

    def _rollout(self, design: np.ndarray, cfg: dict, return_trace: bool = False):
        # TODO 5: run PyBullet rollout and compute objectives
        # Required outputs:
        # - final tracking error [m]
        # - actuation energy [J]
        # Optional trace for rendering: ee path, error curve, torque trace
        raise NotImplementedError('Implement _rollout')

    def simulate(self, design: np.ndarray, config: dict | None = None) -> np.ndarray:
        # TODO 6: call rollout with cfg merge and clipping to design bounds
        raise NotImplementedError('Implement simulate')

    def optimize(self, starting_point: np.ndarray, config: dict | None = None):
        # TODO 7: implement deterministic local search + OptiStep history
        raise NotImplementedError('Implement optimize')

    def render(self, design: np.ndarray, *, open_window: bool = False):
        # TODO 8: create 4-panel interpretation plot
        # Suggested panels: design vars, task-space path, error-vs-time, torque-vs-time
        raise NotImplementedError('Implement render')

    def random_design(self):
        # TODO 9: sample uniformly in design bounds
        raise NotImplementedError('Implement random_design')


### Step 3 - Smoke-test your scaffold

Run minimal checks to verify interface consistency and simulator behavior.


Use the multi-panel render to read **where heat enters**, **how material is distributed**, and **where thermal bottlenecks remain**.


Use the final figure to interpret whether the design/controller combination reaches the target robustly with acceptable energy use.


In [ ]:
# Run this cell after finishing TODOs in PlanarManipulatorCoDesignProblem
problem = PlanarManipulatorCoDesignProblem(
    seed=42,
    target_x=0.9,
    target_y=0.45,
    payload_kg=0.8,
    disturbance_scale=0.04,
    sim_steps=220,
    max_iter=40,
)
start, _ = problem.random_design()

cfg = {
    'target_x': 0.9,
    'target_y': 0.45,
    'payload_kg': 0.8,
    'disturbance_scale': 0.04,
    'sim_steps': 220,
    'dt': 1.0 / 120.0,
    'torque_limit': 12.0,
    'max_iter': 40,
}

print('design space:', problem.design_space)
print('objectives:', problem.objectives)
print('conditions:', problem.conditions)

viol = problem.check_constraints(start, config=cfg)
print('constraint violations:', len(viol))

obj0 = problem.simulate(start, config=cfg)
opt_design, history = problem.optimize(start, config=cfg)
objf = problem.simulate(opt_design, config=cfg)

print('initial objectives [tracking_error_m, energy_J]:', obj0.tolist())
print('final objectives   [tracking_error_m, energy_J]:', objf.tolist())
print('optimization steps:', len(history))
print('How to read plots: vars | task-space path | error timeline | torque timeline')

problem.render(opt_design)


## Mapping to real EngiBench contributions

Translate this robotics co-design scaffold into domain-specific simulators and datasets (robotics, controls, etc.) with documented assumptions.


## Contribution checklist

Before proposing a new problem, verify data provenance, split policy, evaluation protocol, and reporting templates.


## Troubleshooting

If a section fails, do not continue downstream. Fix locally first, then rerun the section and its immediate checks.
This notebook is intentionally staged so failures are localized.


## Takeaways

Before closing, record three points:
1. What conclusion is directly supported by your metrics?
2. What remains uncertain (and why)?
3. What extra experiment would you run next to reduce that uncertainty?
